# 06 邏輯斯迴歸 — 參考解答

松柏護理之家退伍軍人症群聚事件邏輯斯迴歸練習的完整解答。

In [ ]:
# Google Colab setup -- 若在本機執行可跳過此 cell
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
import pathlib

import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# -- CJK font setup (避免中文標籤顯示為方框) --
# 掃描系統字型目錄，顯式註冊 CJK 字型（比依賴快取更可靠）
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False

df = pd.read_csv("data/synthetic/legionella_outbreak.csv")
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)
fs_map = {"bedridden": 0, "wheelchair": 1, "ambulatory": 2}
df["functional_score"] = df["functional_status"].map(fs_map)

## 題目 1：死亡預測 — Crude OR

In [ ]:
# 建立結果變項
df["dead"] = (df["outcome"] == "dead").astype(int)

# 嚴重度轉數值（只對感染者有意義，未感染者設為 0）
sev_map = {"not_ill": 0, "asymptomatic": 0, "mild": 1, "moderate": 2, "severe": 3}
df["severity_score"] = df["clinical_severity"].map(sev_map)

# 只用感染者做死亡預測（未感染者不會死於此疾病）
cases = df[df["infected"] == 1].copy()
print(f"感染者：{len(cases)} 人，死亡：{cases['dead'].sum()} 人")

# Crude OR
factors_death = ["age", "comorbidity_chf", "comorbidity_copd",
                 "immunosuppressed", "severity_score"]

crude_death = []
for var in factors_death:
    model = smf.logit(f"dead ~ {var}", data=cases).fit(disp=0)
    coef = model.params[var]
    ci = model.conf_int().loc[var]
    crude_death.append({
        "variable": var,
        "crude_OR": round(np.exp(coef), 3),
        "95% CI": f"{np.exp(ci[0]):.3f}\u2013{np.exp(ci[1]):.3f}",
        "p-value": round(model.pvalues[var], 4),
    })

crude_death_df = pd.DataFrame(crude_death)
print("\n=== 死亡預測：Crude OR ===")
print(crude_death_df.to_string(index=False))

## 題目 2：多變項模型

In [ ]:
formula_death = (
    "dead ~ age + comorbidity_chf + comorbidity_copd + "
    "immunosuppressed + severity_score"
)
model_death = smf.logit(formula_death, data=cases).fit(disp=0)

# Adjusted OR 表格
adj_death = []
for var in model_death.params.index:
    if var == "Intercept":
        continue
    coef = model_death.params[var]
    ci = model_death.conf_int().loc[var]
    adj_death.append({
        "variable": var,
        "adjusted_OR": round(np.exp(coef), 3),
        "95% CI": f"{np.exp(ci[0]):.3f}\u2013{np.exp(ci[1]):.3f}",
        "p-value": round(model_death.pvalues[var], 4),
    })

adj_death_df = pd.DataFrame(adj_death)
print("=== 死亡預測：Adjusted OR ===")
print(adj_death_df.to_string(index=False))

# Crude vs Adjusted 比較
print("\n=== Crude vs Adjusted ===")
for var in factors_death:
    c_row = crude_death_df[crude_death_df["variable"] == var].iloc[0]
    a_row = adj_death_df[adj_death_df["variable"] == var]
    if len(a_row) == 0:
        continue
    a_row = a_row.iloc[0]
    change = (a_row["adjusted_OR"] - c_row["crude_OR"]) / c_row["crude_OR"] * 100
    print(f"  {var:25s}  crude={c_row['crude_OR']:.3f}  "
          f"adj={a_row['adjusted_OR']:.3f}  ({change:+.1f}%)")

## 題目 3（挑戰題）：模型比較 + 森林圖

In [ ]:
# 模型 A（精簡）
model_a = smf.logit(
    "dead ~ age + immunosuppressed + severity_score",
    data=cases
).fit(disp=0)

# 模型 B（完整）
model_b = smf.logit(
    "dead ~ age + comorbidity_chf + comorbidity_copd + "
    "immunosuppressed + severity_score",
    data=cases
).fit(disp=0)

print("=== 模型比較 ===")
print(f"  模型 A（3 變項）AIC = {model_a.aic:.1f}")
print(f"  模型 B（5 變項）AIC = {model_b.aic:.1f}")

best = model_a if model_a.aic < model_b.aic else model_b
best_name = "A" if model_a.aic < model_b.aic else "B"
print(f"  \u2192 模型 {best_name} 較佳（AIC 較小）")

In [ ]:
# 森林圖（用較好的模型）
forest_data = []
for var in best.params.index:
    if var == "Intercept":
        continue
    coef = best.params[var]
    ci = best.conf_int().loc[var]
    forest_data.append({
        "variable": var,
        "OR": np.exp(coef),
        "ci_lo": np.exp(ci[0]),
        "ci_hi": np.exp(ci[1]),
    })

fdf = pd.DataFrame(forest_data)

fig, ax = plt.subplots(figsize=(8, 4))
y_pos = range(len(fdf))
ax.errorbar(
    fdf["OR"], y_pos,
    xerr=[fdf["OR"] - fdf["ci_lo"], fdf["ci_hi"] - fdf["OR"]],
    fmt="o", color="#e34a33", capsize=4, markersize=8,
)
ax.axvline(x=1, color="gray", linestyle="--", alpha=0.5)
ax.set_yticks(list(y_pos))
ax.set_yticklabels(fdf["variable"])
ax.set_xlabel("Adjusted Odds Ratio")
ax.set_title(f"\u6b7b\u4ea1\u9810\u6e2c\u6a21\u578b {best_name} \u2014 Adjusted OR \u68ee\u6797\u5716")
plt.tight_layout()
plt.show()

### 解讀

- **severity_score**：臨床嚴重度是死亡最強的預測因子（OR 最大），這符合直覺
- **immunosuppressed**：控制嚴重度後，免疫抑制可能仍為獨立危險因子
- **age**：年齡每增加一歲的 OR 看起來接近 1，但累積效應大（例如 80 歲 vs 70 歲差 10 歲）
- **模型選擇**：AIC 較小的模型不一定每個變項都顯著，但整體平衡較好
- **限制**：死亡人數只有 ~19 人，模型自由度有限，不宜放太多變項